# Notebook 01 — EDA & Cohort Definition
**E. coli Levofloxacin — Index-S Cohort**

Replicates `script6/00_ecoli_lev_cohort.R`  
Validation target: `output6/02_ecoli_lev_cohort_summary.csv` (6,230 patients, 19,374 obs)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

DATA_DIR   = Path('c:/ARMD/data')
OUT_DIR    = Path('c:/ARMD/Python_Coursework_ARMD_analysis/python_output')
R_OUT_DIR  = Path('c:/ARMD/output6')  # for validation
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_GAP_DAYS   = 7
TARGET_AB      = 'Levofloxacin'
TARGET_ORG     = 'ESCHERICHIA COLI'
STATE_ORDER    = ['Susceptible', 'Intermediate', 'Resistant']
STATE_COLORS   = {'Susceptible': '#2166ac', 'Intermediate': '#fee090', 'Resistant': '#d73027'}

print('Setup complete.')

## 1. Load Raw Data


In [ ]:
cols = ['anon_id', 'order_proc_id_coded', 'order_time_jittered_utc',
        'culture_description', 'was_positive', 'organism',
        'antibiotic', 'susceptibility']

raw = pd.read_csv(DATA_DIR / 'microbiology_cultures_cohort.csv', usecols=cols)
print(f'Raw rows loaded: {len(raw):,}')

## 2. Filter to E. coli UTI

In [ ]:
ec = raw[
    raw['organism'].str.contains(TARGET_ORG, case=False, na=False) &
    raw['culture_description'].str.contains('URINE', case=False, na=False) &
    (raw['was_positive'] == 1) &
    raw['susceptibility'].isin(STATE_ORDER)
].copy()

print(f'E. coli UTI records: {len(ec):,}')

# Parse order time
ec['order_time'] = pd.to_datetime(ec['order_time_jittered_utc'], utc=True)

## 3. Conflict Resolution
Same order × antibiotic with multiple results → keep most resistant (R > I > S).

In [ ]:
res_rank = {'Resistant': 0, 'Intermediate': 1, 'Susceptible': 2}
ec['res_rank'] = ec['susceptibility'].map(res_rank)

ec = (
    ec.sort_values('res_rank')
      .groupby(['anon_id', 'order_proc_id_coded', 'antibiotic'], as_index=False)
      .first()
      .drop(columns='res_rank')
)

print(f'After conflict resolution: {len(ec):,}')

## 4. Build Levofloxacin Sequences
Apply 7-day gap filter, require ≥ 2 observations, index state = Susceptible.

In [ ]:
lev = ec[ec['antibiotic'] == TARGET_AB].copy()
lev = lev.sort_values(['anon_id', 'order_time']).reset_index(drop=True)

# Compute interval between consecutive observations per patient
lev['interval_days'] = (
    lev.groupby('anon_id')['order_time']
       .diff()
       .dt.total_seconds() / 86400
)

# Drop rows where gap to previous obs is < 7 days
lev = lev[lev['interval_days'].isna() | (lev['interval_days'] >= MIN_GAP_DAYS)].copy()

# Keep patients with >= 2 observations
n_obs = lev.groupby('anon_id').size()
lev = lev[lev['anon_id'].isin(n_obs[n_obs >= 2].index)].copy()

# Index-S filter: first observation must be Susceptible
first_state = lev.groupby('anon_id')['susceptibility'].first()
index_s_patients = first_state[first_state == 'Susceptible'].index
lev = lev[lev['anon_id'].isin(index_s_patients)].copy()

# Recalculate interval_days after filtering
lev['interval_days'] = (
    lev.groupby('anon_id')['order_time']
       .diff()
       .dt.total_seconds() / 86400
)

n_pts = lev['anon_id'].nunique()
n_obs_total = len(lev)
print(f'Cohort: {n_pts:,} patients, {n_obs_total:,} observations')

## 5. Cohort Characterization

In [ ]:
followup = (
    lev.groupby('anon_id')['order_time']
       .agg(first='min', last='max')
)
followup['followup_days'] = (followup['last'] - followup['first']).dt.total_seconds() / 86400

obs_counts = lev.groupby('anon_id').size()

summary = {
    'organism': 'ESCHERICHIA COLI',
    'antibiotic': 'Levofloxacin',
    'index_state': 'Susceptible',
    'n_patients': n_pts,
    'n_observations': n_obs_total,
    'median_followup_days': round(followup['followup_days'].median(), 3),
    'q25_followup_days': round(followup['followup_days'].quantile(0.25), 3),
    'q75_followup_days': round(followup['followup_days'].quantile(0.75), 3),
    'max_followup_days': round(followup['followup_days'].max(), 3),
    'n_obs_group_2': (obs_counts == 2).sum(),
    'n_obs_group_3_5': obs_counts.between(3, 5).sum(),
    'n_obs_group_6_10': obs_counts.between(6, 10).sum(),
    'n_obs_group_gt10': (obs_counts > 10).sum(),
}

summary_df = pd.DataFrame(list(summary.items()), columns=['metric', 'value'])
summary_df.to_csv(OUT_DIR / '01_ecoli_lev_cohort_summary.csv', index=False)
print(summary_df.to_string(index=False))

### Validation vs R

In [ ]:
r_summary = pd.read_csv(R_OUT_DIR / '02_ecoli_lev_cohort_summary.csv')
print('--- R output ---')
print(r_summary.to_string(index=False))
print(f"\nPython n_patients: {n_pts:,} | R: {r_summary.loc[r_summary.metric=='n_patients','value'].values[0]}")
print(f"Python n_obs:      {n_obs_total:,} | R: {r_summary.loc[r_summary.metric=='n_observations','value'].values[0]}")

## 6. Transition Matrix

In [ ]:
lev['from_state'] = lev.groupby('anon_id')['susceptibility'].shift(1)
lev['to_state']   = lev['susceptibility']
trans = lev.dropna(subset=['from_state']).copy()

# Count transitions
counts = trans.groupby(['from_state', 'to_state']).size().reset_index(name='N')
totals = counts.groupby('from_state')['N'].transform('sum')
counts['total'] = totals
counts['proportion'] = (counts['N'] / counts['total']).round(4)
counts['transition'] = counts['from_state'].str[0] + '->' + counts['to_state'].str[0]

counts = counts[['transition', 'from_state', 'to_state', 'N', 'total', 'proportion']]
counts.to_csv(OUT_DIR / '01_ecoli_lev_transition_matrix.csv', index=False)
print(counts.to_string(index=False))

## 7. Sojourn Times

In [ ]:
def sojourn_stats(grp, label_col, label_val):
    d = grp['interval_days'].dropna()
    return pd.Series({
        'group': label_col,
        'label': label_val,
        'N': len(d),
        'mean': round(d.mean(), 1),
        'median': round(d.median(), 1),
        'sd': round(d.std(), 1),
        'q25': round(d.quantile(0.25), 1),
        'q75': round(d.quantile(0.75), 1),
        'min': round(d.min(), 1),
        'max': round(d.max(), 1),
    })

by_state = [sojourn_stats(g, 'by_from_state', name)
            for name, g in trans.groupby('from_state')]

trans['transition'] = trans['transition'] if 'transition' in trans.columns else (
    trans['from_state'].str[0] + '->' + trans['to_state'].str[0]
)
by_trans = [sojourn_stats(g, 'by_transition', name)
            for name, g in trans.groupby('transition')]

sojourn_df = pd.DataFrame(by_state + by_trans)
sojourn_df = sojourn_df[sojourn_df['N'] >= 5]  # suppress sparse transitions
sojourn_df.to_csv(OUT_DIR / '01_ecoli_lev_sojourn_summary.csv', index=False)
print(sojourn_df.to_string(index=False))

## 8. Transition Heatmap

In [ ]:
# Build pivot: rows = from_state, cols = to_state
pivot_n = counts.pivot(index='from_state', columns='to_state', values='N').fillna(0)
pivot_p = counts.pivot(index='from_state', columns='to_state', values='proportion').fillna(0)

# Reorder to S / I / R
pivot_n = pivot_n.reindex(index=STATE_ORDER, columns=STATE_ORDER).fillna(0)
pivot_p = pivot_p.reindex(index=STATE_ORDER, columns=STATE_ORDER).fillna(0)

# Annotation: 'N\n(P%)'
annot = pd.DataFrame(
    [[f"{int(pivot_n.iloc[r,c])}\n({pivot_p.iloc[r,c]*100:.1f}%)"
      for c in range(3)] for r in range(3)],
    index=STATE_ORDER, columns=STATE_ORDER
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot_p, annot=annot, fmt='', cmap='Blues',
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Proportion'})
ax.set_title('E. coli Levofloxacin — Transition Proportions', fontsize=13)
ax.set_xlabel('To state')
ax.set_ylabel('From state')
plt.tight_layout()
plt.savefig(OUT_DIR / '01_ecoli_lev_transition_heatmap.png', dpi=150)
plt.show()

## 9. Sojourn Boxplot

In [ ]:
# Only transitions with N >= 5
valid_trans = sojourn_df[sojourn_df['group'] == 'by_transition']['label'].tolist()

trans['transition_label'] = trans['from_state'].str[0] + '->' + trans['to_state'].str[0]
plot_data = trans[trans['transition_label'].isin(valid_trans) &
                  trans['interval_days'].notna()].copy()

medians = plot_data.groupby('transition_label')['interval_days'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=plot_data, x='transition_label', y='interval_days',
            order=medians.index, palette='Set2', ax=ax,
            flierprops={'marker': '.', 'alpha': 0.3})
ax.set_yscale('log')
ax.set_ylabel('Interval (days, log scale)')
ax.set_xlabel('Transition')
ax.set_title('Sojourn Times by Transition — E. coli Levofloxacin')

# Median labels above boxes
for i, (tlab, med) in enumerate(medians.items()):
    ax.text(i, med * 1.4, f'{med:.0f}d', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / '01_ecoli_lev_sojourn_boxplot.png', dpi=150)
plt.show()

## 10. Susceptibility Distribution by Visit Index

In [ ]:
lev['obs_index'] = lev.groupby('anon_id').cumcount() + 1
lev['obs_index_capped'] = lev['obs_index'].clip(upper=10)  # cap at 10 for display

visit_counts = (
    lev.groupby(['obs_index_capped', 'susceptibility'])
       .size().reset_index(name='n')
)
visit_total = lev.groupby('obs_index_capped').size().reset_index(name='total')
visit_counts = visit_counts.merge(visit_total, on='obs_index_capped')
visit_counts['prop'] = visit_counts['n'] / visit_counts['total']

pivot = visit_counts.pivot(index='obs_index_capped', columns='susceptibility', values='prop').fillna(0)
pivot = pivot.reindex(columns=STATE_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(kind='bar', stacked=True, ax=ax,
           color=[STATE_COLORS[s] for s in STATE_ORDER], edgecolor='white')
ax.set_xlabel('Visit index (capped at 10)')
ax.set_ylabel('Proportion')
ax.set_title('Susceptibility Distribution by Visit — E. coli Levofloxacin')
ax.legend(title='State', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / '01_ecoli_lev_susceptibility_by_visit.png', dpi=150)
plt.show()

## 11. Save Cohort Dataset for Notebook 02

In [ ]:
save_cols = ['anon_id', 'order_proc_id_coded', 'order_time',
             'susceptibility', 'obs_index', 'interval_days',
             'from_state', 'to_state']
lev[save_cols].to_csv(OUT_DIR / '01_ecoli_lev_cohort_dataset.csv', index=False)
print(f'Saved cohort dataset: {len(lev):,} rows')